In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import pytz

def show_stacked_area_chart(csv_file_path):
    # 1. Check Current Time in IST
    ist_tz = pytz.timezone('Asia/Kolkata')
    current_time_ist = datetime.now(ist_tz)
    
    # 2. Time-Gate Logic: Only execute between 16:00 (4 PM) and 18:00 (6 PM)
    if not (16 <= current_time_ist.hour < 18):
        print(f"Graph is currently hidden. It is only available between 4 PM and 6 PM IST. (Current time: {current_time_ist.strftime('%I:%M %p')} IST)")
        return  # Exit the function

    # 3. Load Data
    df = pd.read_csv(csv_file_path)

    # 4. Clean and Transform Columns
    # Clean Rating
    df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
    
    # Clean Reviews
    df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

    # Clean Size (Extract numeric MB)
    def clean_size_mb(size):
        if isinstance(size, str):
            if 'M' in size:
                return float(size.replace('M', ''))
            elif 'k' in size:
                return float(size.replace('k', '')) / 1024 # Convert kb to MB
        return np.nan
    df['Size_MB'] = df['Size'].apply(clean_size_mb)

    # Clean Installs (Convert to numeric)
    df['Installs'] = df['Installs'].astype(str).str.replace(r'[+,]', '', regex=True)
    df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

    # Clean Last Updated
    df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')

    # 5. Apply Complex Filters
    # Rating >= 4.2
    df = df[df['Rating'] >= 4.2]
    
    # App name must NOT contain any numbers (Regex \d matches any digit)
    df = df[~df['App'].str.contains(r'\d', na=False)]
    
    # Category must start with 'T' or 'P'
    df = df[df['Category'].str.startswith(('T', 'P'), na=False)]
    
    # Reviews > 1,000
    df = df[df['Reviews'] > 1000]
    
    # Size between 20 MB and 80 MB
    df = df[(df['Size_MB'] >= 20.0) & (df['Size_MB'] <= 80.0)]

    # Drop rows missing crucial time or install data
    df = df.dropna(subset=['Last Updated', 'Installs'])

    # 6. Apply Category Translations for the Legend
    translations = {
        'TRAVEL_AND_LOCAL': 'Voyages et local', # French
        'PRODUCTIVITY': 'Productividad',        # Spanish
        'PHOTOGRAPHY': '写真'                   # Japanese
    }
    df['Category'] = df['Category'].replace(translations)

    # 7. Aggregate Cumulative Installs Over Time
    df['Month'] = df['Last Updated'].dt.to_period('M').dt.to_timestamp()
    
    # Group by Category and Month to get monthly installs
    monthly_df = df.groupby(['Category', 'Month'])['Installs'].sum().reset_index()
    
    # Pivot so Months are the index and Categories are columns
    pivot_df = monthly_df.pivot(index='Month', columns='Category', values='Installs').fillna(0)
    
    # Calculate cumulative sum of installs over time
    cumulative_df = pivot_df.cumsum()

    # 8. Calculate MoM Growth to find Highlight Periods
    # Calculate month-over-month percentage change for the cumulative totals
    mom_growth = cumulative_df.pct_change()
    
    # Find any month where AT LEAST ONE category grew by more than 25% (> 0.25)
    highlight_months = mom_growth[(mom_growth > 0.25).any(axis=1)].index

    # 9. Render the Stacked Area Chart
    fig, ax = plt.subplots(figsize=(12, 7))
    
    categories = cumulative_df.columns
    # Create the stacked area chart
    ax.stackplot(
        cumulative_df.index, 
        [cumulative_df[cat] for cat in categories], 
        labels=categories, 
        alpha=0.8
    )

    # 10. Apply Highlighting Logic
    # Overlay a dark semi-transparent span on months with >25% growth to "increase intensity"
    for month in highlight_months:
        # Add a span from the start of the month to the end of the month (approx 30 days)
        end_of_month = month + pd.Timedelta(days=30)
        ax.axvspan(month, end_of_month, color='black', alpha=0.15, 
                   label='>25% MoM Growth' if month == highlight_months[0] else "")

    # Formatting and Labels
    ax.set_title('Cumulative App Installs Over Time (Filtered Categories)', fontsize=14)
    ax.set_xlabel('Date (Last Updated)', fontsize=12)
    ax.set_ylabel('Cumulative Installs', fontsize=12)
    
    # Clean up duplicate legend entries
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper left')

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show() # Use st.pyplot(fig) if using Streamlit

# Usage
show_stacked_area_chart('googleplaystore.csv')

Graph is currently hidden. It is only available between 4 PM and 6 PM IST. (Current time: 02:29 PM IST)
